### Conduct vector similarity search on Azure OpenAI embeddings using Azure Cache for Redis

https://learn.microsoft.com/en-us/azure/azure-cache-for-redis/cache-tutorial-vector-similarity


In [2]:
# pip install "openai==1.6.1" num2words matplotlib plotly scipy scikit-learn pandas tiktoken redis langchain
%pip install num2words plotly scipy

Defaulting to user installation because normal site-packages is not writeable
  Using cached num2words-0.5.13-py3-none-any.whl.metadata (12 kB)
  Using cached docopt-0.6.2-py2.py3-none-any.whl
Using cached num2words-0.5.13-py3-none-any.whl (143 kB)
   ---------------------------------------- 0.0/19.1 MB ? eta -:--:--
   ---- ----------------------------------- 2.4/19.1 MB 13.4 MB/s eta 0:00:02
   ----------- ---------------------------- 5.2/19.1 MB 13.3 MB/s eta 0:00:02
   ---------------- ----------------------- 7.9/19.1 MB 13.5 MB/s eta 0:00:01
   ------------------- -------------------- 9.4/19.1 MB 11.7 MB/s eta 0:00:01
   ------------------------- -------------- 12.3/19.1 MB 12.2 MB/s eta 0:00:01
   --------------------------------- ------ 15.7/19.1 MB 12.7 MB/s eta 0:00:01
   ---------------------------------------  18.9/19.1 MB 13.1 MB/s eta 0:00:01
   ---------------------------------------- 19.1/19.1 MB 12.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updat

In [ ]:
# Code cell 2
import re
from num2words import num2words
import os
import pandas as pd
import tiktoken
from typing import List
from langchain_openai import AzureOpenAIEmbeddings
from langchain.vectorstores.redis import Redis as RedisVectorStore
from langchain.document_loaders import DataFrameLoader

API_KEY = ""
RESOURCE_ENDPOINT = "https://<>.openai.azure.com/"
DEPLOYMENT_NAME = "text-embedding-ada-002"
MODEL_NAME = "text-embedding-ada-002"
REDIS_ENDPOINT = "<>.westeurope.redisenterprise.cache.azure.net:10000"
REDIS_PASSWORD = ""

Download the dataset
In a web browser, navigate to https://www.kaggle.com/datasets/jrobischon/wikipedia-movie-plots

Import dataset into pandas and process data


In [3]:
# Code cell 3

df=pd.read_csv(os.path.join(os.getcwd(),'archive/wiki_movie_plots_deduped.csv'))
df

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...
...,...,...,...,...,...,...,...,...
34881,2014,The Water Diviner,Turkish,Director: Russell Crowe,Director: Russell Crowe\r\nCast: Russell Crowe...,unknown,https://en.wikipedia.org/wiki/The_Water_Diviner,"The film begins in 1919, just after World War ..."
34882,2017,Çalgı Çengi İkimiz,Turkish,Selçuk Aydemir,"Ahmet Kural, Murat Cemcir",comedy,https://en.wikipedia.org/wiki/%C3%87alg%C4%B1_...,"Two musicians, Salih and Gürkan, described the..."
34883,2017,Olanlar Oldu,Turkish,Hakan Algül,"Ata Demirer, Tuvana Türkay, Ülkü Duru",comedy,https://en.wikipedia.org/wiki/Olanlar_Oldu,"Zafer, a sailor living with his mother Döndü i..."
34884,2017,Non-Transferable,Turkish,Brendan Bradley,"YouTubers Shanna Malcolm, Shira Lazar, Sara Fl...",romantic comedy,https://en.wikipedia.org/wiki/Non-Transferable...,The film centres around a young woman named Am...


Next, process the data by adding an id index, removing spaces from the column titles, and filters the movies to take only movies made after 1970 and from English speaking countries. This filtering step reduces the number of movies in the dataset, which lowers the cost and time required to generate embeddings. You're free to change or remove the filter parameters based on your preferences.


In [4]:
# Code cell 4

df.insert(0, 'id', range(0, len(df)))
df['year'] = df['Release Year'].astype(int)
df['origin'] = df['Origin/Ethnicity'].astype(str)
del df['Release Year']
del df['Origin/Ethnicity']
df = df[df.year > 1970] # only movies made after 1970
df = df[df.origin.isin(['American','British','Canadian'])] # only movies from English-speaking cinema
df

,id,Title,Director,Cast,Genre,Wiki Page,Plot,year,origin
8626,8626,$ aka Dollars,Richard Brooks,"Warren Beatty, Goldie Hawn",unknown,https://en.wikipedia.org/wiki/$_(film),"Set in Hamburg, West Germany, several criminal...",1971,American
8627,8627,200 Motels,"Tony Palmer, Charles Swenson","Frank Zappa, Ringo Starr, Theodore Bikel",unknown,https://en.wikipedia.org/wiki/200_Motels,"In 200 Motels, the film attempts to portray th...",1971,American
8628,8628,The Anderson Tapes,Sidney Lumet,"Sean Connery, Dyan Cannon, Christopher Walken,...",unknown,https://en.wikipedia.org/wiki/The_Anderson_Tapes,"Burglar John ""Duke"" Anderson (Sean Connery) is...",1971,American
8629,8629,The Andromeda Strain,Robert Wise,"Arthur Hill, James Olson, Kate Reid, David Way...",unknown,https://en.wikipedia.org/wiki/The_Andromeda_St...,"After a satellite, a U.S. government project c...",1971,American
8630,8630,Bad Man's River,Eugenio Martin,"Lee Van Cleef, Gina Lollobrigida",unknown,https://en.wikipedia.org/wiki/Bad_Man%27s_River,Roy King's gang robs a bank and flees to Mexic...,1971,American
...,...,...,...,...,...,...,...,...,...
22428,22428,"Hochelaga, Land of Souls (Hochelaga terre des ...",François Girard,"Raoul Max Trujillo, Tanaya Beatty, David La Haye",historical drama,"https://en.wikipedia.org/wiki/Hochelaga,_Land_...","One night on the campus of McGill University, ...",2017,Canadian
22429,22429,Indian Horse,Stephen Campanelli,"Forrest Goodluck, Michiel Huisman, Michael Mur...",drama,https://en.wikipedia.org/wiki/Indian_Horse_(film),"The Indian Horse family, including six-year-ol...",2017,Canadian
22430,22430,The Little Girl Who Was Too Fond of Matches (L...,Simon Lavoie,NaN,unknown,https://en.wikipedia.org/wiki/The_Little_Girl_...,"In rural 1930s Quebec, Alice lives in house wi...",2017,Canadian
22431,22431,Meditation Park,Mina Shum,"Sandra Oh, Liane Balaban, Don McKellar",drama,https://en.wikipedia.org/wiki/Meditation_Park,"Opened by Mandarin theme song, Meditation Park...",2017,Canadian


Create a function to clean the data by removing whitespace and punctuation, then use it against the dataframe containing the plot.


In [5]:
# Code cell 5

pd.options.mode.chained_assignment = None

# s is input text
def normalize_text(s, sep_token = " \n "):
    s = re.sub(r'\s+',  ' ', s).strip()
    s = re.sub(r". ,","",s)
    # remove all instances of multiple spaces
    s = s.replace("..",".")
    s = s.replace(". .",".")
    s = s.replace("\n", "")
    s = s.strip()

    return s

df['Plot']= df['Plot'].apply(lambda x : normalize_text(x))

Finally, remove any entries that contain plot descriptions that are too long for the embeddings model. (In other words, they require more tokens than the 8192 token limit.) and then calculate the numbers of tokens required to generate embeddings. This also impacts pricing for embedding generation.


In [6]:
# Code cell 6

tokenizer = tiktoken.get_encoding("cl100k_base")
df['n_tokens'] = df["Plot"].apply(lambda x: len(tokenizer.encode(x)))
df = df[df.n_tokens<8192]
print('Number of movies: ' + str(len(df)))
print('Number of tokens required:' + str(df['n_tokens'].sum()))

Number of movies: 11125
Number of tokens required:7044844


### Load DataFrame into LangChain


In [7]:
# Code cell 7

loader = DataFrameLoader(df, page_content_column="Plot" )
movie_list = loader.load()

### Generate embeddings and load them into Redis


In [12]:
# Code cell 8

embedding = AzureOpenAIEmbeddings(
    deployment=DEPLOYMENT_NAME,
    model=MODEL_NAME,
    azure_endpoint=RESOURCE_ENDPOINT,
    openai_api_type="azure",
    openai_api_key=API_KEY,
    openai_api_version="2023-05-15",
    show_progress_bar=True,
    chunk_size=16 # current limit with Azure OpenAI service. This will likely increase in the future.
    )

# name of the Redis search index to create
index_name = "movieindex"

# create a connection string for the Redis Vector Store. Uses Redis-py format: https://redis-py.readthedocs.io/en/stable/connections.html#redis.Redis.from_url
# This example assumes TLS is enabled. If not, use "redis://" instead of "rediss://
redis_url = "rediss://:" + REDIS_PASSWORD + "@"+ REDIS_ENDPOINT

# create and load redis with documents
vectorstore = RedisVectorStore.from_documents(
    documents=movie_list,
    embedding=embedding,
    index_name=index_name,
    redis_url=redis_url
)

# save index schema so you can reload in the future without re-generating embeddings
vectorstore.write_schema("redis_schema.yaml")

  0%|          | 0/696 [00:00<?, ?it/s]

### Run vector search queries


In [15]:
# Code cell 9

query = "Spaceships, aliens, and heroes saving America"
results = vectorstore.similarity_search_with_score(query, k=10)

for i, j in enumerate(results):
    movie_title = str(results[i][0].metadata['Title'])
    similarity_score = str(round((1 - results[i][1]),4))
    print(movie_title + ' (Score: ' + similarity_score + ')')

  0%|          | 0/1 [00:00<?, ?it/s]

Independence Day (Score: 0.8334)
The Flying Machine (Score: 0.8315)
Bravestarr: The Legend (Score: 0.8289)
Remote Control (Score: 0.8285)
Invaders from Mars (Score: 0.8277)
Xenogenesis (Score: 0.8275)
Apocalypse Earth (Score: 0.8274)
Thru the Moebius Strip (Score: 0.827)
Invasion from Inner Earth (Score: 0.8267)
Solar Crisis (Score: 0.8264)


### Hybrid searches

Since RediSearch also features rich search functionality on top of vector search, it's possible to filter results by the metadata in the data set, such as film genre, cast, release year, or director. In this case, filter based on the genre comedy.


In [16]:
# Code cell 10

from langchain.vectorstores.redis import RedisText

query = "Spaceships, aliens, and heroes saving America"
genre_filter = RedisText("Genre") == "comedy"
results = vectorstore.similarity_search_with_score(query, filter=genre_filter, k=10)
for i, j in enumerate(results):
    movie_title = str(results[i][0].metadata['Title'])
    similarity_score = str(round((1 - results[i][1]),4))
    print(movie_title + ' (Score: ' + similarity_score + ')')

  0%|          | 0/1 [00:00<?, ?it/s]

Remote Control (Score: 0.8285)
Meet Dave (Score: 0.8222)
Elf-Man (Score: 0.8194)
Mars Attacks! (Score: 0.8151)
Fifty/Fifty (Score: 0.8148)
Strange Invaders (Score: 0.8129)
Amanda and the Alien (Score: 0.8124)
Coneheads (Score: 0.8121)
Suburban Commando (Score: 0.8119)
Morons from Outer Space (Score: 0.8107)
